# Experiment 5: Sparsification

Trains Caltech256 64x64 isotropic MLPs for 25 epochs and then measures the effect of full alternating sparsification on accuracy and parameter count over 5 repeats.

Scaled epsilon follows:

$$\epsilon_{\mathrm{scaled}} = \frac{100}{N}\sum_i \frac{\lVert z_{\mathrm{post}}^{(i)}-z_{\mathrm{pre}}^{(i)}\rVert_2}{\lVert z_{\mathrm{pre}}^{(i)}\rVert_2}$$

KL divergence follows:

$$D_{\mathrm{KL}} = \frac{1}{N}\sum_i D_{\mathrm{KL}}(\operatorname{softmax}(z_{\mathrm{pre}}^{(i)}) \| \operatorname{softmax}(z_{\mathrm{post}}^{(i)}))$$


In [ ]:
import copy
import pickle as pkl
from pathlib import Path
import numpy as np
import pandas as pd
import time
import math
import random
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchvision.datasets.utils import download_and_extract_archive
from torch.utils.data import DataLoader, Subset
import platform
from Dependencies import *

In [ ]:
ARCHITECTURES = [
    [3 * 64 * 64, 500, 500, 257],
    [3 * 64 * 64, 500, 500, 500, 257],
    [3 * 64 * 64, 500, 500, 500, 500, 257],
    [3 * 64 * 64, 500, 500, 500, 500, 500, 257],
    [3 * 64 * 64, 500, 500, 500, 500, 500, 500, 257],
    [3 * 64 * 64, 1000, 1000, 257],
    [3 * 64 * 64, 1000, 1000, 1000, 257],
    [3 * 64 * 64, 1000, 1000, 1000, 1000, 257],
    [3 * 64 * 64, 1000, 1000, 1000, 1000, 1000, 257],
    [3 * 64 * 64, 1000, 1000, 1000, 1000, 1000, 1000, 257],
]

REPEATS = 5
TOTAL_EPOCHS = 25


LEARNING_RATE = 1e-3
BATCH_SIZE = 48
PSI_DECAY = 1e-1
ADAMW_WEIGHT_DECAY = 1e-3
WEIGHT_INIT = "orthogonal"

INTRINSIC_LENGTH_APPROACH = "TRAINABLE"
LINEAR_CORRECTION_APPROACH = "TRAINABLE+DECAY"
POSITIVE_INTRINSIC_LENGTH = True
INIT_INTRINSIC_LENGTH = 1e-6
TANH_EPSILON = 1e-3

# Non-zero parameter threshold
PARAMETER_EPSILON_THRESHOLD = 1e-4

DATA_ROOT = Path("./data")
NORMALISATION = True
CALTECH256_IMAGE_SIZE = 64
CALTECH256_TRAIN_FRACTION = 0.8
CALTECH256_SPLIT_SEED = 20260504
CALTECH256_NORMALISATION_PATH = Path(
    f"./CALTECH256_{CALTECH256_IMAGE_SIZE}x{CALTECH256_IMAGE_SIZE}_normalisations.pkl"
)

# Matched repeat seeds across architectures
SEED_GENERATOR_SEED = 123450
REPEAT_SEEDS = np.random.default_rng(SEED_GENERATOR_SEED).integers(
    low=0,
    high=2**31 - 1,
    size=REPEATS,
    dtype=np.int64,
).tolist()

DEVICE = try_gpu(output=True, i=0)

SAVE_DIR = Path("./FinalResults/Exp5Results/Caltech256-64x64/")
DATA_DIR = SAVE_DIR / "data"
MODEL_SAVE_DIR = SAVE_DIR / "trained_models"
ACTIVATION_SAVE_DIR = SAVE_DIR / "test_set_activations"
CACHE_TRAINED_MODELS = True
SAVE_TEST_SET_ACTIVATION_TENSORS = True

SAVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
ACTIVATION_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment 5 settings")
print("---------------------")
print(f"Architectures: {ARCHITECTURES}")
print(f"Repeats per architecture: {REPEATS}")
print(f"Repeat seeds: {REPEAT_SEEDS}")
print(f"Training epochs per repeat: {TOTAL_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Parameter count threshold: abs(parameter) > {PARAMETER_EPSILON_THRESHOLD}")
print(f"Model approaches: intrinsic={INTRINSIC_LENGTH_APPROACH}, linear correction={LINEAR_CORRECTION_APPROACH}")


In [ ]:
# -------------------------
# Runtime/GPU facts
# -------------------------

def print_runtime_facts(device):
    print("Runtime facts")
    print("-------------")
    print(f"Python: {platform.python_version()}")
    print(f"PyTorch: {torch.__version__}")
    print(f"Platform: {platform.platform()}")
    print(f"Selected device: {device}")

    if torch.cuda.is_available() and str(device).startswith("cuda"):
        idx = device.index if device.index is not None else 0
        props = torch.cuda.get_device_properties(idx)
        print(f"CUDA available: True")
        print(f"CUDA version used by PyTorch: {torch.version.cuda}")
        print(f"GPU name: {torch.cuda.get_device_name(idx)}")
        print(f"GPU total memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"GPU allocated memory now: {torch.cuda.memory_allocated(idx) / 1024**3:.3f} GB")
        print(f"GPU reserved memory now: {torch.cuda.memory_reserved(idx) / 1024**3:.3f} GB")
    else:
        print("CUDA available: False")

print_runtime_facts(DEVICE)

In [ ]:
# -------------------------
# Caltech256 64x64 data
# -------------------------

class RequiredPerPixelNormalize:
    def __init__(self, path):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(
                f"Required normalisation file not found: {path}. "
                "This experiment intentionally does not create normalisation .pkl files."
            )
        normaliser_dictionary = pkl.load(open(path, "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor):
        return (tensor - self.mean) * self.inv_std

CALTECH256_ARCHIVE_MD5 = "67b4f42ca05d46448c6bb8ecd2220f6d"
CALTECH256_DOWNLOAD_URLS = [
    "https://data.caltech.edu/records/nyy15-4j048/files/256_ObjectCategories.tar?download=1",
    "http://www.vision.caltech.edu/Image_Datasets/Caltech256/256_ObjectCategories.tar",
]

def ensure_caltech256_files_available():
    caltech_root = DATA_ROOT / "caltech256"
    categories_dir = caltech_root / "256_ObjectCategories"
    if categories_dir.exists():
        return

    caltech_root.mkdir(parents=True, exist_ok=True)
    download_errors = []
    for url in CALTECH256_DOWNLOAD_URLS:
        try:
            download_and_extract_archive(
                url,
                download_root=str(caltech_root),
                filename="256_ObjectCategories.tar",
                md5=CALTECH256_ARCHIVE_MD5,
            )
            if categories_dir.exists():
                return
        except Exception as err:
            download_errors.append(f"{url}: {err}")

    raise RuntimeError(
        "Could not download/extract Caltech256 from any configured URL. "
        "You can also manually place 256_ObjectCategories under ./data/caltech256/.\n"
        + "\n".join(download_errors)
    )

def make_caltech256_transform():
    steps = [
        transforms.Lambda(lambda image: image.convert("RGB")),
        transforms.Resize((CALTECH256_IMAGE_SIZE, CALTECH256_IMAGE_SIZE)),
        transforms.ToTensor(),
    ]
    if NORMALISATION:
        steps.append(RequiredPerPixelNormalize(CALTECH256_NORMALISATION_PATH))
    return transforms.Compose(steps)

def make_caltech256_split_indices(dataset_length):
    rng = np.random.default_rng(CALTECH256_SPLIT_SEED)
    indices = rng.permutation(dataset_length)
    train_count = int(round(CALTECH256_TRAIN_FRACTION * dataset_length))
    return indices[:train_count].tolist(), indices[train_count:].tolist()

if NORMALISATION:
    print(f"Using normalisation: {CALTECH256_NORMALISATION_PATH}")
else:
    print("Not using normalisation")

ensure_caltech256_files_available()
transform = make_caltech256_transform()

caltech256_full = datasets.Caltech256(
    root=str(DATA_ROOT),
    download=False,
    transform=transform,
)
train_indices, test_indices = make_caltech256_split_indices(len(caltech256_full))
caltech256_train = Subset(caltech256_full, train_indices)
caltech256_test = Subset(caltech256_full, test_indices)

train_loader = DataLoader(caltech256_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(caltech256_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Caltech256 64x64: train={len(caltech256_train)}, test={len(caltech256_test)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")


In [ ]:
# -------------------------
# Helper functions
# -------------------------

def set_repeat_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def architecture_key(architecture):
    return "x".join(str(v) for v in architecture)

def model_cache_path(architecture, repeat_idx, seed):
    return MODEL_SAVE_DIR / (
        f"architecture_{architecture_key(architecture)}_"
        f"repeat_{repeat_idx:02d}_seed_{int(seed)}_{TOTAL_EPOCHS}epochs.pkl"
    )

def build_network(architecture):
    network = IsotropicTanhMLP(
        layers=architecture,
        flatten=True,
        unflatten_shape=None,
        intrinsic_length_approach=INTRINSIC_LENGTH_APPROACH,
        linear_correction_approach=LINEAR_CORRECTION_APPROACH,
        positive_intrinsic_length=POSITIVE_INTRINSIC_LENGTH,
        init_intrinsic_length=INIT_INTRINSIC_LENGTH,
        tanh_epsilon=TANH_EPSILON,
        device=DEVICE,
        dtype=torch.get_default_dtype(),
    )
    network.simple_initialiser(weight_init=WEIGHT_INIT)
    return network

def count_nonzero_parameters(model, threshold=PARAMETER_EPSILON_THRESHOLD):
    total = 0
    with torch.no_grad():
        for parameter in model.parameters():
            total += int((parameter.detach().abs() > threshold).sum().item())
    return total

def total_parameter_slots(model):
    return int(sum(parameter.numel() for parameter in model.parameters()))

@torch.no_grad()
def predict_test_set_outputs(model, dataloader, device):
    model.eval()
    logits_list = []
    labels_list = []

    for batch_data, batch_labels in dataloader:
        batch_data = batch_data.to(device)
        logits = model(batch_data)

        logits_list.append(logits.detach().cpu())
        labels_list.append(batch_labels.detach().cpu())

    return torch.cat(logits_list, dim=0), torch.cat(labels_list, dim=0)

def accuracy_from_logits(logits, labels):
    return 100.0 * (logits.argmax(dim=1) == labels).float().mean().item()

def mean_activation_l2(logits):
    return float(torch.linalg.norm(logits, dim=1).mean().item())

def scaled_epsilon_from_logits(logits_after, logits_before):
    before_norm = torch.linalg.norm(logits_before, dim=1)
    zero_norm = before_norm == 0
    if torch.any(zero_norm):
        raise ZeroDivisionError(
            "Divide-by zero in scale-aware epsilon probably logits having zero L2 norm"
        )
    diff_norm = torch.linalg.norm(logits_after - logits_before, dim=1)
    return (100.0 * diff_norm / before_norm).mean().item()

def kl_divergence_from_logits(logits_after, logits_before):
    p_before = torch.softmax(logits_before, dim=1)
    log_p_after = torch.log_softmax(logits_after, dim=1)
    return torch.nn.functional.kl_div(
        log_p_after,
        p_before,
        reduction="batchmean",
    ).item()

def valid_odd_full_diagonal_layers(architecture):
    # IsotropicTanhMLP.full_diagonalise requires: 1 <= layer < len(architecture)-2.
    # Odd indexed layers only: 1, 3, 5, ...
    return [
        layer
        for layer in range(1, len(architecture) - 2)
        if layer % 2 == 1
    ]

def apply_full_alternating_sparsification(model, architecture, verbose=False):
    # Alternating full diagonalisation
    layers = valid_odd_full_diagonal_layers(architecture)

    for layer in layers:
        if verbose:
            print(f"Applying full_diagonalise(layer={layer})")
        model.full_diagonalise(layer=layer, verbose=verbose)

    return model, layers

def save_activation_tensors(architecture, repeat_idx, seed, before_logits, after_logits, labels):
    if not SAVE_TEST_SET_ACTIVATION_TENSORS:
        return None

    path = ACTIVATION_SAVE_DIR / (
        f"architecture_{architecture_key(architecture)}_"
        f"repeat_{repeat_idx:02d}_seed_{int(seed)}_activations.pt"
    )
    torch.save(
        {
            "architecture": architecture,
            "repeat_idx": repeat_idx,
            "seed": int(seed),
            "before_logits": before_logits,
            "after_logits": after_logits,
            "labels": labels,
            "scaled_epsilon_definition": "100/N sum_i ||z_post-z_pre||_2 / ||z_pre||_2",
            "kl_definition": "mean D_KL(softmax(z_pre) || softmax(z_post))",
        },
        path,
    )
    return str(path)

def train_or_load_model(architecture, repeat_idx, seed):
    path = model_cache_path(architecture, repeat_idx, seed)

    if CACHE_TRAINED_MODELS and path.exists():
        print(f"Loading cached model: {path}")
        with open(path, "rb") as f:
            cached = pkl.load(f)
        return cached["model"].to(DEVICE), cached.get("stats", None), str(path), True

    print(f"Training architecture={architecture}, repeat={repeat_idx}, seed={int(seed)}, epochs={TOTAL_EPOCHS}")
    network = build_network(architecture).to(DEVICE)
    optimiser = torch.optim.AdamW(
        network.parameters(),
        lr=LEARNING_RATE,
        weight_decay=ADAMW_WEIGHT_DECAY,
    )

    trained_network, stats = training_loop(
        network=network,
        training_set=train_loader,
        testing_set=test_loader,
        epochs=TOTAL_EPOCHS,
        learning_rate=LEARNING_RATE,
        device=DEVICE,
        quiet=True,
        optimiser=optimiser,
        classification_or_reconstruction="classification",
        lambda_psi=PSI_DECAY,
    )

    if CACHE_TRAINED_MODELS:
        with open(path, "wb") as f:
            pkl.dump(
                {
                    "architecture": architecture,
                    "repeat_idx": repeat_idx,
                    "seed": int(seed),
                    "epochs": TOTAL_EPOCHS,
                    "model": trained_network,
                    "stats": stats,
                },
                f,
            )
        print(f"Saved trained model to: {path}")

    return trained_network.to(DEVICE), stats, str(path), False


In [ ]:
# -------------------------
# Main experiment
# -------------------------

raw_records = []

for architecture_idx, architecture in enumerate(ARCHITECTURES):
    sparsified_layers = valid_odd_full_diagonal_layers(architecture)

    print("\n" + "=" * 90)
    print(f"Architecture index {architecture_idx}: {architecture}")
    print(f"Valid odd full-diagonal sparsification layers: {sparsified_layers if sparsified_layers else 'none'}")
    print("=" * 90)

    for repeat_idx in range(REPEATS):
        seed = REPEAT_SEEDS[repeat_idx]
        set_repeat_seed(seed)

        network, stats, cached_model_path, loaded_from_cache = train_or_load_model(architecture, repeat_idx, seed)
        network = network.to(DEVICE)
        network.eval()

        # Before sparsification
        before_logits, labels = predict_test_set_outputs(network, test_loader, DEVICE)
        before_accuracy = accuracy_from_logits(before_logits, labels)
        before_parameter_count = count_nonzero_parameters(network)
        before_parameter_slots = total_parameter_slots(network)
        before_activation_mean_l2 = mean_activation_l2(before_logits)

        # Sparsify clone
        sparsified_network = copy.deepcopy(network).to(DEVICE)
        sparsification_start_time = time.perf_counter()
        sparsified_network, layers_applied = apply_full_alternating_sparsification(
            sparsified_network,
            architecture=architecture,
            verbose=False,
        )
        sparsification_elapsed_seconds = time.perf_counter() - sparsification_start_time

        # After sparsification
        after_logits, labels_after = predict_test_set_outputs(sparsified_network, test_loader, DEVICE)
        if not torch.equal(labels, labels_after):
            raise RuntimeError("Test labels changed between pre/post sparsification evaluation.")

        after_accuracy = accuracy_from_logits(after_logits, labels_after)
        after_parameter_count = count_nonzero_parameters(sparsified_network)
        after_parameter_slots = total_parameter_slots(sparsified_network)
        after_activation_mean_l2 = mean_activation_l2(after_logits)

        scaled_epsilon = scaled_epsilon_from_logits(after_logits, before_logits)
        kl_divergence = kl_divergence_from_logits(after_logits, before_logits)
        delta_accuracy = after_accuracy - before_accuracy
        delta_parameter_count = after_parameter_count - before_parameter_count

        activation_tensor_path = save_activation_tensors(
            architecture=architecture,
            repeat_idx=repeat_idx,
            seed=seed,
            before_logits=before_logits,
            after_logits=after_logits,
            labels=labels,
        )

        record = {
            "architecture_idx": architecture_idx,
            "architecture": str(architecture),
            "repeat_idx": repeat_idx,
            "seed": int(seed),
            "epochs": TOTAL_EPOCHS,
            "loaded_from_cache": loaded_from_cache,
            "model_cache_path": cached_model_path,
            "sparsified_layers": str(layers_applied),
            "pre_accuracy": before_accuracy,
            "post_accuracy": after_accuracy,
            "delta_accuracy": delta_accuracy,
            "pre_nonzero_parameter_count": before_parameter_count,
            "post_nonzero_parameter_count": after_parameter_count,
            "delta_nonzero_parameter_count": delta_parameter_count,
            "pre_parameter_slots": before_parameter_slots,
            "post_parameter_slots": after_parameter_slots,
            "pre_activation_mean_l2": before_activation_mean_l2,
            "post_activation_mean_l2": after_activation_mean_l2,
            "scaled_epsilon_percent": scaled_epsilon,
            "kl_divergence_nats": kl_divergence,
            "sparsification_elapsed_seconds": sparsification_elapsed_seconds,
            "activation_tensor_path": activation_tensor_path,
        }
        raw_records.append(record)

        print(
            f"repeat={repeat_idx:02d} | "
            f"acc {before_accuracy:.4f}% -> {after_accuracy:.4f}% "
            f"(Δ={delta_accuracy:+.6f}) | "
            f"scaled epsilon={scaled_epsilon:.8f}% | "
            f"KL={kl_divergence:.8f} | "
            f"|θ|>{PARAMETER_EPSILON_THRESHOLD}: {before_parameter_count} -> {after_parameter_count} "
            f"(Δ={delta_parameter_count:+d}) | "
            f"layers={layers_applied}"
        )

        del network, sparsified_network, before_logits, after_logits, labels, labels_after
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

raw_df = pd.DataFrame(raw_records)
display(raw_df)


In [ ]:
# -------------------------
# Architecture-level summary
# -------------------------

def safe_se(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size <= 1:
        return 0.0
    return float(np.std(values, ddof=1) / np.sqrt(values.size))

def safe_ratio(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)

    ratio = np.full_like(numerator, np.nan, dtype=np.float64)
    valid = denominator != 0
    ratio[valid] = numerator[valid] / denominator[valid]
    return ratio

raw_df = raw_df.copy()

raw_df["nonzero_parameter_retention_fraction"] = safe_ratio(
    raw_df["post_nonzero_parameter_count"],
    raw_df["pre_nonzero_parameter_count"],
)

raw_df["nonzero_parameter_retention_percent"] = (
    100.0 * raw_df["nonzero_parameter_retention_fraction"]
)

raw_df["nonzero_parameter_reduction_fraction"] = (
    1.0 - raw_df["nonzero_parameter_retention_fraction"]
)

raw_df["nonzero_parameter_reduction_percent"] = (
    100.0 * raw_df["nonzero_parameter_reduction_fraction"]
)

raw_df["delta_nonzero_parameter_count"] = (
    raw_df["post_nonzero_parameter_count"] - raw_df["pre_nonzero_parameter_count"]
)

raw_df["delta_nonzero_parameter_count_percent_of_pre"] = (
    100.0
    * safe_ratio(
        raw_df["delta_nonzero_parameter_count"],
        raw_df["pre_nonzero_parameter_count"],
    )
)

summary_rows = []

for (architecture_idx, architecture), group in raw_df.groupby(
    ["architecture_idx", "architecture"],
    sort=True,
):
    summary_rows.append({
        "architecture_idx": architecture_idx,
        "architecture": architecture,
        "repeats": len(group),
        "sparsified_layers": group["sparsified_layers"].iloc[0],

        "pre_accuracy_mean": float(group["pre_accuracy"].mean()),
        "pre_accuracy_se": safe_se(group["pre_accuracy"]),
        "post_accuracy_mean": float(group["post_accuracy"].mean()),
        "post_accuracy_se": safe_se(group["post_accuracy"]),
        "delta_accuracy_mean": float(group["delta_accuracy"].mean()),
        "delta_accuracy_se": safe_se(group["delta_accuracy"]),

        "pre_nonzero_parameter_count_mean": float(group["pre_nonzero_parameter_count"].mean()),
        "pre_nonzero_parameter_count_se": safe_se(group["pre_nonzero_parameter_count"]),
        "post_nonzero_parameter_count_mean": float(group["post_nonzero_parameter_count"].mean()),
        "post_nonzero_parameter_count_se": safe_se(group["post_nonzero_parameter_count"]),

        "delta_nonzero_parameter_count_mean": float(group["delta_nonzero_parameter_count"].mean()),
        "delta_nonzero_parameter_count_se": safe_se(group["delta_nonzero_parameter_count"]),

        "delta_nonzero_parameter_count_percent_of_pre_mean": float(
            group["delta_nonzero_parameter_count_percent_of_pre"].mean()
        ),
        "delta_nonzero_parameter_count_percent_of_pre_se": safe_se(
            group["delta_nonzero_parameter_count_percent_of_pre"]
        ),

        "nonzero_parameter_retention_percent_mean": float(
            group["nonzero_parameter_retention_percent"].mean()
        ),
        "nonzero_parameter_retention_percent_se": safe_se(
            group["nonzero_parameter_retention_percent"]
        ),

        "nonzero_parameter_reduction_percent_mean": float(
            group["nonzero_parameter_reduction_percent"].mean()
        ),
        "nonzero_parameter_reduction_percent_se": safe_se(
            group["nonzero_parameter_reduction_percent"]
        ),

        "pre_parameter_slots": int(group["pre_parameter_slots"].iloc[0]),
        "post_parameter_slots": int(group["post_parameter_slots"].iloc[0]),

        "pre_activation_mean_l2_mean": float(group["pre_activation_mean_l2"].mean()),
        "pre_activation_mean_l2_se": safe_se(group["pre_activation_mean_l2"]),
        "post_activation_mean_l2_mean": float(group["post_activation_mean_l2"].mean()),
        "post_activation_mean_l2_se": safe_se(group["post_activation_mean_l2"]),

        "scaled_epsilon_percent_mean": float(group["scaled_epsilon_percent"].mean()),
        "scaled_epsilon_percent_se": safe_se(group["scaled_epsilon_percent"]),
        "kl_divergence_nats_mean": float(group["kl_divergence_nats"].mean()),
        "kl_divergence_nats_se": safe_se(group["kl_divergence_nats"]),

        "sparsification_elapsed_seconds_mean": float(group["sparsification_elapsed_seconds"].mean()),
        "sparsification_elapsed_seconds_se": safe_se(group["sparsification_elapsed_seconds"]),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

raw_csv_path = DATA_DIR / "experiment_5_sparsification_raw_repeats.csv"
summary_csv_path = DATA_DIR / "experiment_5_sparsification_summary.csv"

raw_df.to_csv(raw_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)

print(f"Saved raw per-repeat table to: {raw_csv_path}")
print(f"Saved architecture-level summary to: {summary_csv_path}")


In [ ]:
# -------------------------
# Compact formatted table
# -------------------------

formatted_rows = []

for _, row in summary_df.iterrows():
    formatted_rows.append({
        "architecture_idx": row["architecture_idx"],
        "architecture": row["architecture"],
        "odd full-diagonal layers": row["sparsified_layers"],

        "accuracy before (%)": (
            f"{row['pre_accuracy_mean']:.4f} ± {row['pre_accuracy_se']:.4f}"
        ),
        "accuracy after (%)": (
            f"{row['post_accuracy_mean']:.4f} ± {row['post_accuracy_se']:.4f}"
        ),
        "delta accuracy (pp)": (
            f"{row['delta_accuracy_mean']:+.6f} ± {row['delta_accuracy_se']:.6f}"
        ),

        f"nonzero params before (>|{PARAMETER_EPSILON_THRESHOLD}|)": (
            f"{row['pre_nonzero_parameter_count_mean']:.1f} ± "
            f"{row['pre_nonzero_parameter_count_se']:.1f}"
        ),
        f"nonzero params after (>|{PARAMETER_EPSILON_THRESHOLD}|)": (
            f"{row['post_nonzero_parameter_count_mean']:.1f} ± "
            f"{row['post_nonzero_parameter_count_se']:.1f}"
        ),

        "post/pre nonzero params (%)": (
            f"{row['nonzero_parameter_retention_percent_mean']:.4f} ± "
            f"{row['nonzero_parameter_retention_percent_se']:.4f}"
        ),

        "nonzero params removed (%)": (
            f"{row['nonzero_parameter_reduction_percent_mean']:.4f} ± "
            f"{row['nonzero_parameter_reduction_percent_se']:.4f}"
        ),

        "scaled epsilon (%)": (
            f"{row['scaled_epsilon_percent_mean']:.8f} ± "
            f"{row['scaled_epsilon_percent_se']:.8f}"
        ),

        "KL divergence (nats)": (
            f"{row['kl_divergence_nats_mean']:.8f} ± "
            f"{row['kl_divergence_nats_se']:.8f}"
        ),
    })

formatted_summary_df = pd.DataFrame(formatted_rows)
display(formatted_summary_df)

formatted_csv_path = DATA_DIR / "experiment_5_sparsification_formatted_summary.csv"
formatted_summary_df.to_csv(formatted_csv_path, index=False)

print(f"Saved formatted summary to: {formatted_csv_path}")


In [ ]:
# -------------------------
# LaTeX table export
# -------------------------

tex_path = SAVE_DIR / "sparsification_summary.tex"

def latex_escape(s):
    s = str(s)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)
    return s

def fmt_mean_pm(mean, se, decimals=4, signed=False):
    if pd.isna(mean):
        return r"--"

    sign = "+" if signed else ""
    mean_s = f"{mean:{sign}.{decimals}f}"
    se_s = f"{se:.{decimals}f}" if not pd.isna(se) else "0"

    return rf"${mean_s} \pm {se_s}$"

rows = []

for _, row in summary_df.sort_values("architecture_idx").iterrows():
    rows.append(
        {
            "Architecture": latex_escape(row["architecture"]),
            "Layers": latex_escape(row["sparsified_layers"]),
            r"$\Delta$ Acc. (pp)": fmt_mean_pm(
                row["delta_accuracy_mean"],
                row["delta_accuracy_se"],
                decimals=6,
                signed=True,
            ),
            r"Params removed (\%)": fmt_mean_pm(
                row["nonzero_parameter_reduction_percent_mean"],
                row["nonzero_parameter_reduction_percent_se"],
                decimals=3,
            ),
            r"$\epsilon_{\mathrm{scaled}}$ (\%)": fmt_mean_pm(
                row["scaled_epsilon_percent_mean"],
                row["scaled_epsilon_percent_se"],
                decimals=6,
            ),
            "KL (nats)": fmt_mean_pm(
                row["kl_divergence_nats_mean"],
                row["kl_divergence_nats_se"],
                decimals=6,
            ),
        }
    )

table_df = pd.DataFrame(rows)

column_format = (
    r"p{0.30\linewidth}"
    r"p{0.10\linewidth}"
    "c"
    "c"
    "c"
    "c"
)

latex_body = table_df.to_latex(
    index=False,
    escape=False,
    column_format=column_format,
)

caption = (
    f"Caltech256 64x64 sparsification by alternating full diagonalisation. "
    f"Values are mean $\\pm$ standard error over {REPEATS} repeats. "
    f"$\\Delta$ accuracy is post-sparsification minus pre-sparsification accuracy in percentage points; "
    f"parameter removal uses $|\\theta|>{PARAMETER_EPSILON_THRESHOLD}$; "
    f"scaled epsilon is reported as a percentage and KL divergence in nats."
)

label = "tab:experiment_5_sparsification_caltech256_64x64"

latex_table = latex_body.replace(
    r"\begin{tabular}",
    "\\begin{table}[ht]\n"
    "\\centering\n"
    "\\scriptsize\n"
    r"\begin{tabular}",
)

latex_table = latex_table.replace(
    r"\end{tabular}",
    r"\end{tabular}"
    + "\n"
    + rf"\caption{{{caption}}}"
    + "\n"
    + rf"\label{{{label}}}"
    + "\n"
    + r"\end{table}",
)

tex_path.write_text(latex_table, encoding="utf-8")

print(f"Saved LaTeX table to: {tex_path}")
print(latex_table)
